# 改错题 · sklearn 数据泄漏

**数据泄漏（data leakage）** 是竞赛里最隐蔽的失分点：代码一行不错，指标还很好看，
但模型一上真实数据就崩 —— 因为它「偷看」了测试集。

这类错误有个共同特征：**它不会报错**。只能靠你对流程顺序的敏感度发现。


## 排查手法

问自己一句：**「这一步用到的信息，在真实预测时能拿到吗？」**

- 标准化用的均值/方差 —— 来自训练集还是全量？
- 编码器的类别表 —— 见过测试集的类别吗？
- 切分之后，测试集有没有再被「碰」过？
- 时序数据，测试集在训练集的**之后**吗？

> 改完对 `_solution.ipynb`，每题都写了「为什么」和「怎么自查」。


### 题 1 · 划分之前就标准化了

先 `fit_transform` 全量数据，再划分训练/测试集。

跑一遍看看：不报错，一切正常 —— 但这种「正常」正是问题所在。

**要求**：把顺序改成「先划分，再 fit 训练集」。

In [ ]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
X = np.column_stack([rng.normal(0, 1, 200), rng.normal(100, 15, 200)])
y = (X[:, 1] > 100).astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.35, random_state=42
)
print("划分完成", X_train.shape, X_test.shape)


In [ ]:

# 修复后请在这里重写（可以直接改上面那个 cell，也可以写在这里）


### 题 2 · 用全量数据 fit 编码器

训练集只有 A~D，测试集里有 E。

跑一遍看看：编码器学到的类别是 `['A','B','C','D','E']` —— **测试集的 E 泄漏了**。

In [ ]:

import numpy as np
from sklearn.preprocessing import OneHotEncoder

cats = np.array([["A"], ["B"], ["A"], ["C"], ["D"], ["B"], ["E"], ["A"]])
train_idx = [0, 1, 2, 3, 4]
test_idx = [5, 6, 7]

encoder = OneHotEncoder(handle_unknown="ignore")
encoder.fit(cats)

print("编码器学到的类别：", encoder.categories_[0].tolist())


In [ ]:

# 修复后请在这里重写（可以直接改上面那个 cell，也可以写在这里）


### 题 3 · 在测试集上误用 fit_transform

先用训练集 fit 了 scaler，然后对测试集调用了 `fit_transform`。

跑一遍看看：测试集缩放后均值被压成 `0`，而 `scaler.mean_` 也变成了测试集的均值。

In [ ]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
X = np.column_stack([rng.normal(0, 1, 200), rng.normal(100, 15, 200)])

X_train, X_test = train_test_split(X, test_size=0.35, random_state=42)

scaler = StandardScaler().fit(X_train)
X_test_scaled = scaler.fit_transform(X_test)

print("测试集缩放后均值：", np.round(X_test_scaled.mean(axis=0), 6))
print("scaler 现在的均值：", np.round(scaler.mean_, 4))


In [ ]:

# 修复后请在这里重写（可以直接改上面那个 cell，也可以写在这里）


### 题 4 · 时序数据被随机打乱切分

对一条按日期排好的负荷曲线调用 `train_test_split`。

跑一遍看看：测试集的时间跨度 `01-01 ~ 04-01` **落在训练集 `01-02 ~ 04-10` 内部**。

In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ts = pd.DataFrame({
    "时间": pd.date_range("2025-01-01", periods=100),
    "负荷": np.random.default_rng(42).normal(100, 10, 100),
})

train, test = train_test_split(ts, test_size=0.25, random_state=42)

print("训练集：", train["时间"].min().date(), "~", train["时间"].max().date())
print("测试集：", test["时间"].min().date(), "~", test["时间"].max().date())


In [ ]:

# 修复后请在这里重写（可以直接改上面那个 cell，也可以写在这里）
